# 05_tuning_2. 튜닝 이어서 — 19절부터

**왜 파일을 나눴는가**: `05_tuning.ipynb`가 18절까지 진행되며 실행 결과(출력)까지 쌓여 파일이 커졌다(180개 이상 셀) — 편집 도구가 파일 전체를 한 번에 읽지 못하는 상황이 생겨, 민석님 요청대로 19절부터는 이 파일에 이어서 쓴다. 로드맵상 새로운 단계가 아니라 **같은 8단계(튜닝)의 연속**이라 절 번호는 19부터 이어서 매기고(`## 19.`), `reports/05_tuning.md`도 그대로 이어서 쓴다(리포트 파일은 나누지 않음).

**0절(이어받기 설정)**: `05_tuning.ipynb` 0~18절은 이미 실행 완료된 상태다. 그 절들을 여기서 처음부터 다시 탐색하지 않고, **이미 확정된 값**(τ=0.70, `DEFAULT_PARAMS`, MLP 구조·`T_soft`=0.003, 그룹별 블렌드 가중치)을 상수로 그대로 가져와 최소한의 코드로 재현한다. 각 셀에 "몇 절에서 확정된 값인지" 출처를 남겨, 나중에 05_tuning.ipynb와 대조할 수 있게 한다.

## 0. 이어받기 설정

### 0-1. 셋업 + 데이터 로드

`05_tuning.ipynb` 0~1절과 동일 — v2 피처셋(50개)을 그대로 불러온다.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

SEED = 42
np.random.seed(SEED)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "data").exists(), "REPO_ROOT를 찾지 못했습니다. 노트북 실행 위치를 확인하세요."

sys.path.insert(0, str(REPO_ROOT))
from src.metric import metric, TARGET_COLS, CAPACITY_KWH

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
pd.set_option("display.max_columns", 100)

train_features = pd.read_parquet(PROCESSED_DIR / "train_features_v2.parquet")
test_features = pd.read_parquet(PROCESSED_DIR / "test_features_v2.parquet")

DROP_COLS = {"forecast_kst_dtm", "forecast_id", *TARGET_COLS}
FEATURE_COLS = [c for c in train_features.columns if c not in DROP_COLS]

print("python executable:", sys.executable)
print("train_features:", train_features.shape, "| 피처 개수:", len(FEATURE_COLS))


python executable: d:\공모전\wind_forecast\venv\Scripts\python.exe
train_features: (26304, 54) | 피처 개수: 50


### 0-2. 확장 윈도우 3-fold CV

`05_tuning.ipynb` 2절과 동일한 fold 정의.

In [2]:
FOLDS = [
    {"name": "fold1", "train_start": "2022-01-01 01:00:00", "train_end": "2023-07-01 00:00:00", "valid_end": "2024-01-01 00:00:00"},
    {"name": "fold2", "train_start": "2022-01-01 01:00:00", "train_end": "2024-01-01 00:00:00", "valid_end": "2024-07-01 00:00:00"},
    {"name": "fold3", "train_start": "2022-01-01 01:00:00", "train_end": "2024-07-01 00:00:00", "valid_end": "2025-01-01 00:00:00"},
]


def make_fold_frames(fold):
    dtm = train_features["forecast_kst_dtm"]
    train_start = pd.Timestamp(fold["train_start"])
    train_end = pd.Timestamp(fold["train_end"])
    valid_end = pd.Timestamp(fold["valid_end"])
    valid_start = train_end + pd.Timedelta(hours=1)

    fold_train = train_features[(dtm >= train_start) & (dtm <= train_end)].reset_index(drop=True)
    fold_valid = train_features[(dtm >= valid_start) & (dtm <= valid_end)].reset_index(drop=True)
    return fold_train, fold_valid


for fold in FOLDS:
    ft, fv = make_fold_frames(fold)
    print(f"{fold['name']}: train {ft.shape}, valid {fv.shape}")


fold1: train (13104, 54), valid (4416, 54)
fold2: train (17520, 54), valid (4368, 54)
fold3: train (21888, 54), valid (4416, 54)


### 0-3. CatBoost 학습·예측·채점 헬퍼 (3·9·11절 압축)

`to_long_ext`(9절), `train_fold_model`/`predict_group`(11-2), `compute_group_metrics`/`combined_score`(11-1)를 그대로 재현한다. `DEFAULT_PARAMS`와 `best_tau=0.70`은 9절에서 이미 확정된 값이라 여기서는 재탐색하지 않고 상수로 둔다(9-2·9-3 참고: τ=0.70, actual 가중, seed 3개 평균 0.6213).

In [3]:
GROUP_ID_MAP = {"kpx_group_1": 0, "kpx_group_2": 1, "kpx_group_3": 2}
GROUP_ID_CATEGORIES = [0, 1, 2]

DEFAULT_PARAMS = {"iterations": 2000, "learning_rate": 0.05}
best_tau = 0.70  # 9절에서 확정(actual 가중 + τ=0.70, seed 3개 평균 tau_seed_mean=0.6213)
tau_seed_mean = 0.6213  # 9-3 기록값 — 19절에서 비교 기준으로만 사용, 재계산하지 않음


def to_long_ext(df, feature_cols):
    frames = []
    for g in TARGET_COLS:
        sub = df[df[g].notna()].copy()
        sub["group_id"] = GROUP_ID_MAP[g]
        sub["utilization"] = sub[g] / CAPACITY_KWH[g]
        sub["actual_kwh"] = sub[g]
        frames.append(sub[["forecast_kst_dtm", "group_id", "utilization", "actual_kwh"] + feature_cols])
    return pd.concat(frames, ignore_index=True)


def make_answer_df(df):
    return df[["forecast_kst_dtm", *TARGET_COLS]].reset_index(drop=True)


def make_pred_df(df, pred_dict):
    out = df[["forecast_kst_dtm"]].reset_index(drop=True).copy()
    for col in TARGET_COLS:
        out[col] = np.clip(pred_dict[col], 0, CAPACITY_KWH[col])
    return out


def train_fold_model(
    fold_train, params, feature_cols=FEATURE_COLS, early_stop_frac=0.15, seed=SEED,
    quantile_alpha=None, use_sample_weight=False, group_weight_multiplier=None,
):
    features = feature_cols + ["group_id"]
    long_df = to_long_ext(fold_train, feature_cols)
    long_df["group_id"] = pd.Categorical(long_df["group_id"], categories=GROUP_ID_CATEGORIES)
    long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_sorted) * early_stop_frac)
    fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]

    loss_function = f"Quantile:alpha={quantile_alpha}" if quantile_alpha is not None else "MAE"
    model = CatBoostRegressor(loss_function=loss_function, random_seed=seed, verbose=False, **params)

    weight = fit_rows["actual_kwh"].to_numpy(dtype=float) if use_sample_weight else None
    if group_weight_multiplier:
        base = weight if weight is not None else np.ones(len(fit_rows))
        mult = fit_rows["group_id"].astype(int).map(group_weight_multiplier).fillna(1.0).to_numpy(dtype=float)
        weight = base * mult

    fit_kwargs = {"sample_weight": weight} if weight is not None else {}
    model.fit(
        fit_rows[features], fit_rows["utilization"],
        eval_set=(early_rows[features], early_rows["utilization"]),
        cat_features=["group_id"], early_stopping_rounds=100, verbose=False,
        **fit_kwargs,
    )
    return model


def predict_group(model, fold_valid, g, feature_cols=FEATURE_COLS):
    features = feature_cols + ["group_id"]
    valid_g = fold_valid.copy()
    valid_g["group_id"] = pd.Categorical([GROUP_ID_MAP[g]] * len(valid_g), categories=GROUP_ID_CATEGORIES)
    return model.predict(valid_g[features]) * CAPACITY_KWH[g]


def compute_group_metrics(answer_df, pred_df):
    result = {}
    for col in TARGET_COLS:
        actual = answer_df[col].to_numpy(dtype=float)
        forecast = pred_df[col].to_numpy(dtype=float)
        capacity = CAPACITY_KWH[col]

        valid = actual >= capacity * 0.10
        actual, forecast = actual[valid], forecast[valid]

        error_rate = np.abs(forecast - actual) / capacity
        nmae = float(np.mean(error_rate))

        unit_price = np.select([error_rate <= 0.06, error_rate <= 0.08], [4.0, 3.0], default=0.0)
        ficr = float(np.sum(actual * unit_price) / np.sum(actual * 4.0))

        result[col] = {"nmae": nmae, "ficr": ficr}
    return result


def combined_score(group_metrics):
    nmaes = [group_metrics[g]["nmae"] for g in TARGET_COLS]
    ficrs = [group_metrics[g]["ficr"] for g in TARGET_COLS]
    one_minus_nmae = 1 - np.mean(nmaes)
    ficr = np.mean(ficrs)
    return 0.5 * one_minus_nmae + 0.5 * ficr, one_minus_nmae, ficr


def single_group_score(fold_valid, g, pred_kwh):
    """단일 그룹만 metric.py 로직대로 직접 채점 (14-8과 동일)."""
    actual = fold_valid[g].to_numpy(dtype=float)
    pred = np.asarray(pred_kwh, dtype=float)
    capacity = CAPACITY_KWH[g]

    valid = actual >= capacity * 0.10
    actual_v, pred_v = actual[valid], pred[valid]

    error_rate = np.abs(pred_v - actual_v) / capacity
    nmae = float(np.mean(error_rate))

    unit_price = np.select([error_rate <= 0.06, error_rate <= 0.08], [4.0, 3.0], default=0.0)
    ficr = float(np.sum(actual_v * unit_price) / np.sum(actual_v * 4.0))
    return nmae, ficr


# 검산: 05_tuning.ipynb 11-1과 같은 방식으로 그룹별 분해 함수가 metric()과 일치하는지 확인
_ft, _fv = make_fold_frames(FOLDS[0])
_model = train_fold_model(_ft, DEFAULT_PARAMS, quantile_alpha=best_tau, use_sample_weight=True)
_pred = {g: predict_group(_model, _fv, g) for g in TARGET_COLS}
_answer_df = make_answer_df(_fv)
_pred_df = make_pred_df(_fv, _pred)
_score_direct, _, _ = metric(_answer_df, _pred_df)
_score_recon, _, _ = combined_score(compute_group_metrics(_answer_df, _pred_df))
print(f"metric() 직접 계산: {_score_direct:.6f} / 그룹별 분해 후 재조합: {_score_recon:.6f}")
assert abs(_score_direct - _score_recon) < 1e-9, "그룹별 분해 로직이 metric()과 불일치합니다"
print("일치 확인 완료 — 이어받기 설정이 05_tuning.ipynb와 같은 결과를 내는 것을 확인")


metric() 직접 계산: 0.603306 / 그룹별 분해 후 재조합: 0.603306
일치 확인 완료 — 이어받기 설정이 05_tuning.ipynb와 같은 결과를 내는 것을 확인


### 0-4. MLP 인프라 (14절 압축)

`build_mlp_features`/`train_mlp_fold`/`predict_group_mlp`/`cv_score_mlp`를 그대로 재현한다. `best_T_soft=0.003`은 14-5·14-6에서 확정된 값(seed 3개 평균 0.6256)이라 재탐색하지 않는다.

In [4]:
import torch

from src import nn as mlp_nn

best_T_soft = 0.003  # 14-5/14-6에서 확정 (mlp_seed_mean=0.6256)
mlp_seed_mean = 0.6256  # 14-6 기록값 — 참고용


def build_mlp_features(df, feature_cols, mu=None, sd=None):
    group_onehot = pd.get_dummies(df["group_id"].astype(int), prefix="grp").reindex(
        columns=[f"grp_{i}" for i in range(3)], fill_value=0
    ).to_numpy(dtype=np.float32)

    num_x = df[feature_cols].fillna(0.0).to_numpy(dtype=np.float64)
    if mu is None:
        mu, sd = mlp_nn.fit_standardizer(num_x)
    num_x = mlp_nn.apply_standardizer(num_x, mu, sd).astype(np.float32)

    x = np.concatenate([num_x, group_onehot], axis=1)
    return x, mu, sd


def train_mlp_fold(
    fold_train, feature_cols, seed=SEED, T_soft=0.003,
    hidden=(256, 256), dropout=0.15, early_stop_frac=0.15, verbose=False,
):
    long_df = to_long_ext(fold_train, feature_cols)
    long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_sorted) * early_stop_frac)
    fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]

    fit_X, mu, sd = build_mlp_features(fit_rows, feature_cols)
    early_X, _, _ = build_mlp_features(early_rows, feature_cols, mu=mu, sd=sd)

    fit_util = fit_rows["utilization"].to_numpy(dtype=np.float32)
    fit_kwh = fit_rows["actual_kwh"].to_numpy(dtype=np.float32)
    fit_scored = fit_util >= 0.10

    early_util = early_rows["utilization"].to_numpy(dtype=np.float32)
    early_kwh = early_rows["actual_kwh"].to_numpy(dtype=np.float32)
    early_scored = early_util >= 0.10

    model, best_val_score, best_epoch = mlp_nn.train_mlp(
        fit_X, fit_util, fit_kwh, fit_scored,
        early_X, early_util, early_kwh, early_scored,
        input_dim=fit_X.shape[1], seed=seed, T_soft=T_soft,
        hidden=hidden, dropout=dropout, verbose=verbose,
    )
    return model, mu, sd, best_epoch


def predict_group_mlp(model, fold_valid, g, feature_cols, mu, sd):
    valid_g = fold_valid.copy()
    valid_g["group_id"] = GROUP_ID_MAP[g]
    x, _, _ = build_mlp_features(valid_g, feature_cols, mu=mu, sd=sd)
    pred_util = mlp_nn.predict_mlp(model, x)
    return pred_util * CAPACITY_KWH[g]


def cv_score_mlp(feature_cols=FEATURE_COLS, seed=SEED, T_soft=0.003, hidden=(256, 256), dropout=0.15, verbose=False):
    scores = []
    for fold in FOLDS:
        fold_train, fold_valid = make_fold_frames(fold)
        model, mu, sd, best_epoch = train_mlp_fold(
            fold_train, feature_cols, seed=seed, T_soft=T_soft, hidden=hidden, dropout=dropout, verbose=verbose,
        )
        pred = {g: predict_group_mlp(model, fold_valid, g, feature_cols, mu, sd) for g in TARGET_COLS}
        answer_df = make_answer_df(fold_valid)
        pred_df = make_pred_df(fold_valid, pred)
        score, _, _ = metric(answer_df, pred_df)
        scores.append(score)
        if verbose:
            print(f"  {fold['name']}: Score={score:.4f} (best_epoch={best_epoch})")
    return float(np.mean(scores)), scores


print("torch version:", torch.__version__)


torch version: 2.13.0+cpu


### 0-5. 확정된 블렌드 가중치 + fold별 예측 캐시 재구성 (14-7·14-8·14-9 압축)

`group_blend_best`(그룹별 CatBoost/MLP 블렌드 가중치: g1=0.4/g2=0.5/g3=1.0)는 14-8에서 확정된 값이라 재탐색하지 않는다. `fold_preds_blend`(fold별 CatBoost·MLP 예측)는 19절에서 신규 후보와 블렌드하려면 실제 예측 배열이 필요해서 여기서 다시 캐시한다(계산 자체는 저비용 — 3-fold 학습 1번씩). `blend_seed_mean`(14-9, seed 3개 평균 CatBoost+MLP 블렌드 점수)은 3개 seed × 3-fold 재학습이 필요해 비용이 크므로, 문서 기록값을 그대로 참고 기준으로만 쓴다.

In [5]:
group_blend_best = {"kpx_group_1": 0.4, "kpx_group_2": 0.5, "kpx_group_3": 1.0}  # 14-8에서 확정
blend_seed_mean = 0.6295  # 14-9 기록값(seed 3개 평균, 표준편차 0.0013) — 19절 비교 기준, 재계산하지 않음

fold_preds_blend = {}
for fold in FOLDS:
    fold_train, fold_valid = make_fold_frames(fold)
    cat_model = train_fold_model(fold_train, DEFAULT_PARAMS, quantile_alpha=best_tau, use_sample_weight=True)
    mlp_model, mlp_mu, mlp_sd, _ = train_mlp_fold(fold_train, FEATURE_COLS, T_soft=best_T_soft)

    cat_pred = {g: predict_group(cat_model, fold_valid, g) for g in TARGET_COLS}
    mlp_pred = {g: predict_group_mlp(mlp_model, fold_valid, g, FEATURE_COLS, mlp_mu, mlp_sd) for g in TARGET_COLS}
    fold_preds_blend[fold["name"]] = {"fold_valid": fold_valid, "cat": cat_pred, "mlp": mlp_pred}

group_blend_fold_scores = []
for fold in FOLDS:
    ctx = fold_preds_blend[fold["name"]]
    pred = {g: (1 - group_blend_best[g]) * ctx["cat"][g] + group_blend_best[g] * ctx["mlp"][g] for g in TARGET_COLS}
    answer_df = make_answer_df(ctx["fold_valid"])
    pred_df = make_pred_df(ctx["fold_valid"], pred)
    score, _, _ = metric(answer_df, pred_df)
    group_blend_fold_scores.append(score)

group_blend_mean = float(np.mean(group_blend_fold_scores))
print(f"기존 2모델(CatBoost+MLP) 그룹별 블렌드 3-fold 평균: {group_blend_mean:.4f} (14-8 기록값 0.6306과 비교)")
print("fold별 CatBoost/MLP 예측 캐시 완료 — 19절에서 재사용")


기존 2모델(CatBoost+MLP) 그룹별 블렌드 3-fold 평균: 0.6306 (14-8 기록값 0.6306과 비교)
fold별 CatBoost/MLP 예측 캐시 완료 — 19절에서 재사용


## 19. 다른 머신러닝/딥러닝 모델 후보 — 블렌드 다양성 확인

지금까지 확정된 최종안은 **CatBoost(τ=0.70+actual가중) + MLP(T_soft=0.003) 그룹별 블렌드**(14절)뿐이다. 11~18절에서 피처·하이퍼파라미터 레버는 거의 소진됐다는 게 확인됐지만(각 절 종합 해석 참고), "블렌드에 넣을 성격이 다른 모델을 하나 더 찾는 것"은 아직 시도하지 않았다. 민석님이 "혹시 모르니까 다른 머신러닝·딥러닝 모델도 써보자"고 제안해 이 절에서 확인한다.

**후보 선정 기준**(민석님과 상의):
- **LightGBM/XGBoost**: 04번에서 이미 CatBoost에게 진 부스팅 계열이라 개별 성능 기대는 낮지만, 저비용이라 확인 가치는 있다(04번 인프라 재사용).
- **RandomForest/ExtraTrees**: 부스팅과 오차 구조가 다른 배깅 계열(여러 트리를 독립적으로 학습해 평균 내는 방식 — 부스팅처럼 이전 트리의 오차를 순차적으로 보정하지 않는다)이라, 블렌드 다양성 확보에 이론적으로 더 유리하다.
- **Attention(작은 Transformer)**: CNN/RNN은 비추천 판단(146번 기록과 동일 근거 — 격자 정보는 03절에서 이미 압축됐고, 시계열 자기회귀는 leakage 제약상 불가)이지만, Attention은 기존 50개 피처를 토큰으로 취급해 `build_mlp_features` 인프라를 거의 그대로 재사용할 수 있어(전처리 재작업 없음) 저비용으로 시도 가능하다고 판단해 포함했다.

**진행 순서**: (1) 5개 후보를 각각 3-fold CV로 단독 성능만 먼저 확인(19-1~19-5) → (2) 트리/배깅 계열 중 최고 성능 후보 하나를 "기존 CatBoost+MLP 블렌드에 3번째 모델로 추가했을 때" 개선되는지 확인(19-6~19-7) → (3) 개선이 확인된 경우만 seed 재검증(19-8) → (4) 종합 해석(19-9).

**19-5 실행 후 결정(민석님과 상의)**: Attention은 3번째 모델 블렌드 테스트(19-6~19-8)에서 제외했다. 단독 CV가 0.6033으로 CatBoost 대비 -0.0180, MLP 대비 -0.0223 — 14절에서 블렌드가 성공했던 MLP는 반대로 단독 성능이 이미 CatBoost보다 좋았다는 점과 대조적이라, 이 정도 격차에서는 블렌드 가중치 최적화가 w3≈0으로 수렴할 가능성이 높다고 판단했다. 게다가 실측 1 에폭 약 42초로 fold별 재학습(19-6)·seed 3개 재검증(19-8)에 들이는 비용이 크다 — 자세한 판단 근거는 19-6 마크다운 참고.

**leakage-guard 체크**: 모든 후보가 9절(`to_long_ext`)·11절(`train_fold_model`/`predict_group`)·14절(`build_mlp_features`)의 fold-safe 구조(각 fold의 학습 구간에서만 표준화·학습, 검증 구간엔 그 결과만 적용)를 그대로 재사용한다 — 새로운 데이터나 미래 정보를 끌어오지 않으므로 별도 점검 없이 기존 판정을 승계한다.

*(이 절은 실패/미채택으로 결론나 코드는 삭제하고 마크다운 기록만 남겼다 — 실행 결과는 각 소절 설명과 마지막 종합 해석, `reports/05_tuning.md`를 참고.)*

### 19-1. LightGBM — τ=0.70 분위수회귀 + actual 가중 (04번 인프라 재사용)

CatBoost 쪽 `train_fold_model`(11-2)과 같은 구조를 LightGBM으로 옮긴다. LightGBM은 `objective="quantile"`+`alpha`로 분위수 회귀를, `categorical_feature`로 `group_id` 범주형을 CatBoost와 동일하게 지원한다. 이번 절 전체에서 재사용할 공용 3-fold CV 러너(`cv_score_generic`)도 여기서 함께 정의한다(모델 종류가 달라도 학습·예측 함수만 바꿔 끼우면 되도록).

### 19-2. XGBoost — τ=0.70 분위수회귀 + actual 가중

XGBoost 2.0 이상은 `objective="reg:quantileerror"`+`quantile_alpha`로 분위수 회귀를 네이티브 지원한다(설치된 버전: 3.3.0). 범주형은 `enable_categorical=True`+pandas category dtype으로 CatBoost·LightGBM과 같은 인코딩을 쓴다.

### 19-3. RandomForest / ExtraTrees — 배깅 계열

**한계를 먼저 밝힌다**: sklearn의 RandomForest/ExtraTrees는 분위수(quantile) 손실을 지원하지 않는다 — CatBoost·LightGBM·XGBoost처럼 τ=0.70으로 예측을 위쪽으로 편향시킬 수 없다. 그래서 이 비교는 "같은 조건에서 어느 모델이 더 나은가"를 보는 공정한 비교가 아니라, "부스팅과 다른 오차 구조(배깅)를 가진 모델이 블렌드에 쓸 만큼 다양성이 있는가"를 보는 탐색적 비교다. actual 가중(표본 가중치)만 동일하게 적용한다.

결측치(`*_diff_prev`류, 발표분 첫 시각에 이전 값이 없어 NaN)는 14-2와 같은 이유로 중립값 0(변화 없음)으로 채운다 — sklearn 트리 구현은 NaN을 직접 처리하지 못한다.

### 19-4. 단독 성능 종합 비교

지금까지 나온 CatBoost(9절)·MLP(14절)·LightGBM·XGBoost·RandomForest·ExtraTrees(위 세 절) 단독 3-fold 평균을 한 표로 본다. **주의**: RandomForest/ExtraTrees는 19-3에서 밝혔듯 quantile 편향이 없어 나머지와 완전히 공정한 비교는 아니다.

### 19-5. Attention(작은 Transformer) — 피처를 토큰으로 취급

**결론 먼저**: 지금까지 시도한 MLP(14·18절)는 모든 피처를 한 벡터로 이어붙여 완전연결층에 통과시킨다 — 피처 사이의 "관계"는 층을 거치며 암묵적으로만 학습된다. Attention(주의 메커니즘 — 문장에서 어떤 단어가 다른 단어와 관련 있는지 가중치를 매기는 것처럼, 여기서는 어떤 피처가 다른 피처와 관련 있는지 가중치를 매기는 방식)은 각 피처를 "토큰"(문장의 단어 하나에 해당하는 단위)으로 취급해서, 피처끼리 서로 얼마나 관련 있는지를 명시적으로 계산한 뒤 값을 예측한다. FT-Transformer(표 데이터 전용 트랜스포머, 2021)의 축소판 구조를 쓴다.

**구현 범위**: `src/nn.py`에 `TabAttention` 클래스를 추가하고, 기존 `train_mlp` 함수가 MLP 대신 이 모델을 받아 그대로 학습할 수 있도록 `model` 인자를 추가했다(안 넘기면 이전과 완전히 동일하게 동작 — 14·18절 재실행 결과에 영향 없음). 학습 루프(미분 가능한 FICR 손실, 조기 종료 등)는 100% 재사용이다.

**실행 시간 경고**: self-attention은 피처 개수의 제곱에 비례해 계산량이 늘어난다(모든 피처 쌍의 관련도를 계산하기 때문). 50개 피처끼리는 크지 않지만 CPU에서 MLP보다 느릴 수 있다. 아래 `ATTN_D_MODEL`·`ATTN_LAYERS`·`ATTN_MAX_EPOCHS`를 작게 잡아 최대한 가볍게 시작했다 — 실행해보고 시간이 오래 걸리면 `ATTN_MAX_EPOCHS`를 줄이거나 이 값들을 더 낮춰서 조절 가능하다.

### 19-6. 3번째 모델 블렌드 후보 준비 — fold별 예측 캐시

트리/배깅 계열(LightGBM·XGBoost·RandomForest·ExtraTrees) 중 19-4에서 CV가 가장 높았던 하나를 "신규후보1"로 선정해 블렌드를 테스트한다.

**Attention은 제외한다(민석님과 상의해 결정)**: 19-5 단독 CV가 0.6033으로 CatBoost(-0.0180)·MLP(-0.0223) 대비 뚜렷이 낮았다. 14절에서 MLP+CatBoost 블렌드가 성공했던 이유는 MLP가 **개별 성능도 이미 CatBoost보다 나았기 때문**(+0.0043)이었다 — "둘 다 잘하는데 오차 종류가 달라서 섞으면 더 좋다"는 구조였다. 지금 Attention은 그 정반대(개별 성능이 뚜렷이 나쁨)라, 블렌드 가중치를 최적화해도 보통 w3(신규 모델 비중)가 0에 가깝게 수렴한다 — 10절 오류분석 결론(오차의 주원인이 예보 자체의 부정확함이라는 공통 한계)을 생각하면 세 모델 다 같은 정보 한계에 부딪히고 있어 "완전히 다른 종류의 오차"가 나올 가능성도 낮다. 게다가 실측 1 에폭 약 42초(self-attention이 배치×헤드×토큰² 크기의 어텐션 행렬을 매 에폭 만들어 배치가 클수록 급격히 느려짐)로, fold별 재학습(이 절)·seed 3개 재검증(19-8)까지 가면 비용이 크다. 낮은 기대 효과 대비 비용이 커서 제외.

14-7에서 이미 캐시해 둔 CatBoost·MLP fold별 예측(`fold_preds_blend`)을 그대로 재사용한다.

### 19-7. 3-모델 블렌드 가중치 스윕

기존 확정 2모델 블렌드(14-8의 `group_blend_best` 그룹별 가중치)는 그대로 두고, 그 예측에 신규후보1(`best_new_name`)을 `w3` 비율만큼 추가로 섞는다: `pred = (1-w3)*기존_2모델_블렌드 + w3*신규후보1`. 그룹별로 독립적으로 `w3`를 탐색한다(11·14절과 같은 원리 — 전체 Score는 그룹별 점수의 단순평균이라 그룹마다 다른 가중치를 써도 된다). `w3=0`이 최적이면 "이 후보는 추가해도 도움이 안 된다"는 뜻이다.

### 19-8. seed 재검증 (조건부)

`model-tuning` 스킬 3절 원칙 — 단일 seed 조합에서 나온 개선은 믿지 않는다. 19-7에서 신규후보1 블렌드가 기존 2모델 블렌드(`group_blend_mean`)를 넘었으면(w3=0이 아닌 조합이 이긴 경우) seed 3개(42/7/2024)로 재검증한다. 비교 기준은 14-9의 `blend_seed_mean`(CatBoost+MLP 2모델 블렌드 seed 평균)이다. 못 넘으면 "3번째 모델 블렌드로는 개선을 찾지 못했다"고 정직하게 기록한다.

### 19-9. 종합 해석

**19-1~19-4, 단독 성능**: LightGBM 0.6101(-0.0112) / XGBoost 0.6112(-0.0101, 트리·배깅 계열 중 최고) / RandomForest 0.6045(-0.0168) / ExtraTrees 0.6046(-0.0167) — CatBoost(0.6213) 대비 전부 열세. 04번에서 이미 CatBoost가 이긴 부스팅 계열이 이번에도 재확인됐고, quantile 손실이 없는 배깅 계열(RF/ET)은 그보다도 더 나빴다.

**19-5, Attention**: 0.6033 — 5종 중 가장 낮음(CatBoost 대비 -0.0180, MLP 대비 -0.0223). 실행도 1 에폭당 약 42초로 다른 후보 대비 압도적으로 느렸다(self-attention이 배치×헤드×토큰² 크기 어텐션 행렬을 매 에폭 만들어 배치가 클수록 급격히 느려짐). 개별 성능이 이 정도로 나쁘면 블렌드 가중치 최적화가 보통 0으로 수렴하고(14절에서 MLP 블렌드가 성공한 건 MLP가 개별로도 이미 CatBoost보다 나았기 때문), 비용 대비 기대효과가 낮다고 판단해 민석님과 상의 후 3번째 모델 블렌드 테스트(19-6~19-8)에서 제외했다.

**19-6~19-7, 3-모델 블렌드(XGBoost)**: 기존 CatBoost+MLP 블렌드(0.6306)에 XGBoost를 그룹별로 다른 비율로 추가 — group_1 10%(+0.0014), group_2 40%(+0.0016), group_3 0%(변화 없음, 이미 MLP 100%라 당연). 종합 3-fold 평균 0.6316(+0.0010).

**19-8, seed 재검증(핵심)**: seed 3개(42/7/2024) 평균 0.6309, 표준편차 0.0010. 기존 2모델 블렌드(blend_seed_mean=0.6295) 대비 **+0.0014, 표준편차의 1.4배** — 이 프로젝트에서 지금까지 "진짜 개선"으로 채택했던 사례(9절 47.6배, 14절 6.2배)에 비하면 훨씬 약한 신호다. 완전한 노이즈(0배 근처, 11절·15절 사례)는 아니지만 자신 있게 뚜렷하다고 말하기도 어려운 애매한 크기였다.

**결론**: 민석님이 이 애매한 신호를 로컬 검증에만 의존하지 않고 **리더보드로 직접 확인**하기로 결정 — CatBoost+MLP+XGBoost 3-모델 블렌드(그룹별 가중치: group_1은 기존 블렌드 90%+XGBoost 10%, group_2는 기존 60%+XGBoost 40%, group_3는 기존 그대로)를 train.ipynb/inference.ipynb에 반영해 재학습·재제출 후 실제 리더보드 점수로 채택 여부를 판단한다.

**이번 절 전체 요약**: 5개 신규 모델(LightGBM/XGBoost/RandomForest/ExtraTrees/Attention) 중 단독으로 CatBoost나 MLP를 이긴 것은 없었다. 그러나 "블렌드 다양성" 관점에서는 XGBoost가 소폭(+0.0010~0.0014)의 개선 신호를 보여, 완전히 소득이 없지는 않았다.

## 20. 외부 파이프라인 대조로 찾은 저비용 신규 피처 — 원본에 있었지만 안 쓰던 컬럼

민석님이 Public 0.63886을 낸 외부 파이프라인(다른 저장소, `D:\공모전\wind-forecast_Competition`)의 실제 코드·설정 파일을 공유해주셨다. 대조해보니 격차의 상당 부분이 **구조 차이**(그룹별 모델 vs 우리 통합모델)와 **피처 리치니스 차이**(179개 vs 우리 50개)에서 온다는 게 드러났다. 구조를 통째로 바꾸는 건 04번 결론(통합모델이 그룹별모델보다 나음, 0.5971 vs 0.5868)을 뒤집는 큰 공사라 신중히 접근하고, 이번 절에서는 **저비용 후보만** 먼저 확인한다.

**민석님과 상의해 고른 후보 — 전부 우리 원본 데이터(`data/processed/train_base_wide.parquet`, GFS 격자)에 이미 있는데 지금까지 어떤 피처에도 안 쓰인 컬럼들**:
- **결빙 점수(icing_score)**: 착빙(과냉각 물방울이 날개에 얼어붙는 현상)이 가장 위험한 기온(-3°C 부근)·고습(80% 이상)에서만 커지도록 만든 연속 점수. `icing_score = exp(-((T-(-3))/4)²) × clip((RH-80)/20, 0, 1)` — 외부 파이프라인(phase2 §2-6)의 정의를 그대로 가져왔다.
- **700/500hPa 풍속·지위고도(gh500)·해면기압(prmsl)·행성경계층 환기율(VRATE)**: 우리는 지금 850hPa까지만 쓰는데(`gfs_ws850hpa`), 더 위쪽 대기(중층 트로프/능선, 종관 규모 기압계)의 정보가 빠져 있었다.
- **850hPa-500hPa 기온차(lapse_850_500)**: 대기 안정도 지표 — 값이 크면(기온이 고도에 따라 빨리 떨어지면) 대류가 활발해 지상 바람이 변동성이 크고, 작으면(역전층) 안정적이라 지상-상층 바람이 잘 안 섞인다.
- **경계층높이 대 허브고도 비율(blh_ratio_hub)**: 이미 있는 `{group}_ldaps_blh`(경계층높이)를 허브고도(117m, `info.xlsx` Hub Height 확인)로 나눈 비율 — 1보다 작으면 "경계층이 허브보다 얕다"는 뜻이고, 이런 조건에서 터빈이 경계층 위의 다른 바람을 맞아 지상 예보가 어긋나기 쉽다(외부 phase2 §2-6).

GFS는 `05_tuning_1.ipynb` 13-1에서 확인한 대로 세 그룹 모두 격자 5 하나만 가중치 1.0으로 쓰므로(공유 피처), 별도 가중조합 없이 격자 5 값을 그대로 쓰면 된다. **leakage-guard**: 전부 같은 발표분의 예보 원자료에서 파생된 값이라 새로운 정보 유입이나 미래 시점 참조가 없다 — 기존 50개 피처와 같은 성격.

**진행**: 두 그룹(A=대기 진단 7개, B=blh_ratio_hub 3개)을 baseline/+A/+B/+A+B 4가지 조합으로 3-fold CV 비교(12·13·16절과 같은 패턴) → 이긴 조합만 seed 3개 재검증 → 종합 해석. 이 절은 05_tuning_2 0-1~0-3(셋업·FOLDS·CatBoost 헬퍼+`best_tau`/`tau_seed_mean`)만 있으면 단독 실행 가능하다(0-4/0-5의 MLP·블렌드 인프라, 19절 불필요).

### 20-1. 신규 피처 생성

GFS 격자 5(공유) 원자료에서 대기 진단 피처 7개(그룹 A)를, 기존 `{group}_ldaps_blh`에서 허브고도 비율(그룹 B) 3개를 만든다.

In [ ]:
raw_train_wide = pd.read_parquet(PROCESSED_DIR / "train_base_wide.parquet")

HUB_HEIGHT_M = 117.0  # info.xlsx Hub Height(m) 확인 (V126/U136 두 모델 모두 117m)


def add_atmos_features(raw_wide, features_df):
    """GFS 격자5(전 그룹 공유, 가중치 1.0 — 05_tuning_1 13-1 GROUP_GRID_WEIGHTS 확인) 원자료에서
    지금까지 어떤 피처에도 안 쓰인 상층대기 진단값 7개를 만든다."""
    t2m_c = raw_wide["gfs_g5_heightAboveGround_2_2t"] - 273.15
    rh2m = raw_wide["gfs_g5_heightAboveGround_2_2r"].clip(upper=100.0)
    icing_score = np.exp(-(((t2m_c - (-3.0)) / 4.0) ** 2)) * ((rh2m - 80.0) / 20.0).clip(0.0, 1.0)

    ws700 = np.sqrt(raw_wide["gfs_g5_isobaricInhPa_700_u"] ** 2 + raw_wide["gfs_g5_isobaricInhPa_700_v"] ** 2)
    ws500 = np.sqrt(raw_wide["gfs_g5_isobaricInhPa_500_u"] ** 2 + raw_wide["gfs_g5_isobaricInhPa_500_v"] ** 2)
    lapse_850_500 = raw_wide["gfs_g5_isobaricInhPa_850_t"] - raw_wide["gfs_g5_isobaricInhPa_500_t"]

    atmos = pd.DataFrame({
        "forecast_kst_dtm": raw_wide["forecast_kst_dtm"],
        "gfs_icing_score": icing_score,
        "gfs_ws700hpa": ws700,
        "gfs_ws500hpa": ws500,
        "gfs_lapse_850_500": lapse_850_500,
        "gfs_gh500": raw_wide["gfs_g5_isobaricInhPa_500_gh"],
        "gfs_prmsl": raw_wide["gfs_g5_meanSea_0_prmsl"],
        "gfs_vrate": raw_wide["gfs_g5_planetaryBoundaryLayer_0_VRATE"],
    })
    merged = features_df.merge(atmos, on="forecast_kst_dtm", how="left")
    assert merged[ATMOS_COLS].isna().sum().sum() == 0, "atmos 피처 병합 후 결측 발생 — merge key 확인 필요"
    return merged


def add_blh_ratio_features(features_df):
    """이미 있는 {group}_ldaps_blh(경계층높이)를 허브고도로 나눈 비율. 새 원자료 불필요."""
    out = features_df.copy()
    for g in TARGET_COLS:
        out[f"{g}_blh_ratio_hub"] = out[f"{g}_ldaps_blh"] / HUB_HEIGHT_M
    return out


ATMOS_COLS = ["gfs_icing_score", "gfs_ws700hpa", "gfs_ws500hpa", "gfs_lapse_850_500", "gfs_gh500", "gfs_prmsl", "gfs_vrate"]
BLH_RATIO_COLS = [f"{g}_blh_ratio_hub" for g in TARGET_COLS]

train_features_atmos = add_atmos_features(raw_train_wide, train_features)
train_features_blhratio = add_blh_ratio_features(train_features)
train_features_both = add_blh_ratio_features(train_features_atmos)

print(f"baseline: {len(FEATURE_COLS)}개")
print(f"+A(대기진단): {len(FEATURE_COLS) + len(ATMOS_COLS)}개")
print(f"+B(blh비율): {len(FEATURE_COLS) + len(BLH_RATIO_COLS)}개")
print(f"+A+B: {len(FEATURE_COLS) + len(ATMOS_COLS) + len(BLH_RATIO_COLS)}개")
train_features_atmos[ATMOS_COLS].describe().T


### 20-2. 4가지 피처셋 조합 3-fold CV 비교

baseline(50) / +A(57, 대기진단) / +B(53, blh비율) / +A+B(60)를 τ=`best_tau`+actual가중(9절 확정 설정)으로 비교한다.

In [ ]:
FEATURE_SETS_20 = {
    "baseline": (train_features, FEATURE_COLS),
    "+A(atmos)": (train_features_atmos, FEATURE_COLS + ATMOS_COLS),
    "+B(blh_ratio)": (train_features_blhratio, FEATURE_COLS + BLH_RATIO_COLS),
    "+A+B": (train_features_both, FEATURE_COLS + ATMOS_COLS + BLH_RATIO_COLS),
}


def cv_score_featureset(df, feature_cols):
    scores = []
    for fold in FOLDS:
        dtm = df["forecast_kst_dtm"]
        fold_train = df[(dtm >= pd.Timestamp(fold["train_start"])) & (dtm <= pd.Timestamp(fold["train_end"]))].reset_index(drop=True)
        fold_valid = df[(dtm >= pd.Timestamp(fold["train_end"]) + pd.Timedelta(hours=1)) & (dtm <= pd.Timestamp(fold["valid_end"]))].reset_index(drop=True)
        model = train_fold_model(fold_train, DEFAULT_PARAMS, feature_cols=feature_cols, quantile_alpha=best_tau, use_sample_weight=True)
        pred = {g: predict_group(model, fold_valid, g, feature_cols=feature_cols) for g in TARGET_COLS}
        answer_df = make_answer_df(fold_valid)
        pred_df = make_pred_df(fold_valid, pred)
        score, _, _ = metric(answer_df, pred_df)
        scores.append(score)
    return float(np.mean(scores)), scores


feature_set_rows = []
for name, (df, cols) in FEATURE_SETS_20.items():
    mean_score, fold_scores = cv_score_featureset(df, cols)
    feature_set_rows.append({"피처셋": name, "피처수": len(cols), "3-fold 평균": mean_score, "fold별": fold_scores})
    print(f"{name}({len(cols)}개): 3-fold 평균={mean_score:.4f} (fold별 {[round(s, 4) for s in fold_scores]})")

feature_set_df = pd.DataFrame(feature_set_rows)
feature_set_df["baseline 대비"] = feature_set_df["3-fold 평균"] - feature_set_df.loc[feature_set_df["피처셋"] == "baseline", "3-fold 평균"].iloc[0]
feature_set_df.sort_values("3-fold 평균", ascending=False).reset_index(drop=True)


### 20-3. 최선 조합 seed 안정성 재검증 (조건부)

`model-tuning` 스킬 3절 원칙 — baseline을 이긴 조합이 있으면 seed 3개(42/7/2024)로 재검증한다. 못 이기면 seed 재검증 없이 "개선 없음"으로 기록한다.

In [ ]:
best_row = feature_set_df.sort_values("3-fold 평균", ascending=False).iloc[0]
baseline_score = feature_set_df.loc[feature_set_df["피처셋"] == "baseline", "3-fold 평균"].iloc[0]

if best_row["피처셋"] == "baseline" or best_row["3-fold 평균"] <= baseline_score:
    print("baseline을 이긴 조합이 없음 — 이번 신규 피처 후보들로는 개선을 찾지 못함, seed 재검증 생략")
    winner_df, winner_cols = None, None
else:
    winner_name = best_row["피처셋"]
    winner_df, winner_cols = FEATURE_SETS_20[winner_name]
    print(f"seed 재검증 대상: {winner_name} (CV {best_row['3-fold 평균']:.4f}, baseline 대비 {best_row['3-fold 평균'] - baseline_score:+.4f})")

    seed_scores_20 = []
    for seed in [42, 7, 2024]:
        fold_scores = []
        for fold in FOLDS:
            dtm = winner_df["forecast_kst_dtm"]
            fold_train = winner_df[(dtm >= pd.Timestamp(fold["train_start"])) & (dtm <= pd.Timestamp(fold["train_end"]))].reset_index(drop=True)
            fold_valid = winner_df[(dtm >= pd.Timestamp(fold["train_end"]) + pd.Timedelta(hours=1)) & (dtm <= pd.Timestamp(fold["valid_end"]))].reset_index(drop=True)
            model = train_fold_model(fold_train, DEFAULT_PARAMS, feature_cols=winner_cols, quantile_alpha=best_tau, use_sample_weight=True, seed=seed)
            pred = {g: predict_group(model, fold_valid, g, feature_cols=winner_cols) for g in TARGET_COLS}
            answer_df = make_answer_df(fold_valid)
            pred_df = make_pred_df(fold_valid, pred)
            score, _, _ = metric(answer_df, pred_df)
            fold_scores.append(score)
        seed_mean = float(np.mean(fold_scores))
        seed_scores_20.append(seed_mean)
        print(f"seed={seed}: 3-fold 평균 Score={seed_mean:.4f} (fold별 {[round(s, 4) for s in fold_scores]})")

    seed_mean_20 = float(np.mean(seed_scores_20))
    seed_std_20 = float(np.std(seed_scores_20))
    print(f"\nseed 3개 평균: {seed_mean_20:.4f}, 표준편차: {seed_std_20:.4f}")
    ratio = (seed_mean_20 - tau_seed_mean) / seed_std_20 if seed_std_20 > 0 else float("nan")
    print(f"baseline(tau_seed_mean={tau_seed_mean:.4f}) 대비 개선폭: {seed_mean_20 - tau_seed_mean:+.4f} (표준편차의 {ratio:.1f}배)")


### 20-4. 종합 해석

(실행 결과 대기 — 민석님이 20-1~20-3을 실행한 뒤 결과를 전달하면 이 자리에 해석을 작성한다.)